## Overview

This notebook addresses the Modelling section of the technical brief. The objective is to build & validate a credit risk model, discuss additional trends, and apply the model to the rejected population (if possible). Furthermore, I suggest ways the credit risk policy could be refined based on the model's outcomes simulating a simple cut-off strategy.

## 1. Set up

This section sets up the notebook environment and project structure required to run the analysis in Colab.

It includes:

* cloning the GitHub repository
* installing the required Python packages
* importing reusable helper functions
* downloading / locating the Lending Club dataset

In [1]:
# clone the repo

%cd /content
!rm -rf silvercat
!git clone https://github.com/dimvasilev/silvercat.git
%cd /content/silvercat

/content
Cloning into 'silvercat'...
remote: Enumerating objects: 68, done.
remote: Counting objects: 100% (68/68), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 68 (delta 22), reused 53 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (68/68), 348.51 KiB | 7.42 MiB/s, done.
Resolving deltas: 100% (22/22), done.
/content/silvercat


In [2]:
# install additional packages

%pip install optuna shap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 19.8 MB/s eta 0:00:00


In [ ]:
# import packages and project helpers

import sys
import importlib
from pathlib import Path

sys.path.insert(0, "/content/silvercat/src")
importlib.invalidate_caches()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from silvercat.utils import setup
from silvercat.modelling import (
    CANDIDATE_VARS,
    load_accepted_modelling_data,
    create_modelling_target,
    prepare_modelling_frame,
    time_split,
    split_summary,
    make_feature_matrices,
    fit_xgb_model,
    score_splits,
    tune_xgb_model,
    xgb_params_from_study,
    performance_summary,
    add_validation_bands,
    gini_by_segment,
    shap_importance_table,
    save_model_artifacts,
    write_inference_helper,
    cumulative_bad_capture,
    plot_bad_capture_curve,
    capture_summary,
    cutoff_summary,
    plot_cutoff_tradeoff,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.float_format", "{:.2f}".format)

In [ ]:
# set up project paths and data

paths = setup(project_root="/content/silvercat")

# 2. Train and validate the model

### 2.1. Preprocessing approach

I restrict the development dataset to the candidate variables identified in section 4 of the 1_analytics notebook.

For simplicity, I use a baseline XGBoost model. XGBoost is more appropriate here than a logistic regression model because it can capture non-linear relationships and interactions between credit-risk variables without requiring manual binning.

Since I plan to use an XGBoost model, there is no further handling of missing values, normalisation or standardisation of variables. Tree-based models are not sensitive to feature scale and XGBoost can handle missingness natively.

Categorical variables are one-hot encoded before model training.

In [ ]:
# candidate quote-stage variables

candidate_vars = CANDIDATE_VARS

In [ ]:
# read accepted applications, restrict to the development window and create the target

accepts_model = load_accepted_modelling_data(paths.accepts, candidate_vars)
accepts_model = create_modelling_target(accepts_model)

accepts_model.shape

### 2.2. Transformations and feature engineering

* `earliest_cr_line` is converted into `account_age_mths`, measuring the age of the borrower’s credit history at loan origination

* `fico_range_low` and `fico_range_high` are combined into `fico_avg`, providing a single FICO score measure

* The heavy-tailed variables identified in section 4 in 1_analytics are converted into binary `0 / 1+` flags. This is not strictly necessary as XGBoost can handle the original numeric values, however these variables are highly skewed, so I decided to transform them as binary flags are more stable and easier to interpret

* Categorical string variables are one-hot encoded before modelling. This is appropriate because XGBoost requires categorical variables to be numeric unless its native categorical support is used. One-hot encoding is simple, transparent and suitable here because the categorical variables have relatively low cardinality

* The original source fields used to create engineered variables are removed to avoid duplication

In [ ]:
# apply feature engineering, one-hot encoding and source-field drops

accepts_model, categorical_cols = prepare_modelling_frame(accepts_model)

accepts_model.head()

In [ ]:
accepts_model.shape

### 2.3 Split the dataset into train, test, oot

* I use a time-based split instead of a random split, as this better reflects how the credit risk model would be used in practice

* The model is trained on earlier originations and then assessed on later originations

* This helps test whether the model generalises over time, which is important for credit risk models because borrower mix, credit policy and economic conditions can change

* The OOT is kept as the final temporal validation sample and is not used to tune the model

In [ ]:
# create train, test and out-of-time samples

splits = time_split(accepts_model)
train, test, oot = splits["train"], splits["test"], splits["oot"]

split_summary(splits)

In [ ]:
# isolate feature matrices and target vectors

X, y, feature_cols = make_feature_matrices(splits)

X_train, y_train = X["train"], y["train"]
X_test, y_test = X["test"], y["test"]
X_oot, y_oot = X["oot"], y["oot"]

X_train.shape, X_test.shape, X_oot.shape

### 2.4. Train a simple baseline XGB model

- I fit the model on the training period Jan 2014 - Jun 2015

- I assess interim performance on Jul 2015 - Dec 2015

- The final OOT validation is Jan 2016 - Jun 2016

In [ ]:
# fit baseline model

xgb_model = fit_xgb_model(
    X_train,
    y_train,
    X_test,
    y_test,
    verbose=50,
)

In [ ]:
# score train, test and OOT samples with the baseline model

splits = score_splits(splits, X, xgb_model, "pbad_baseline")
train, test, oot = splits["train"], splits["test"], splits["oot"]

#### Add a minimal hyperparemeter optimisation layer

- To demonstrate hyperparameter optimisation, I created a simple study using Optuna, a bayesian-based tuning package. See [optuna docs](https://optuna.org/)

- I use this as a lightweight tuning layer rather than a heavy search.

- The OOT sample is not used in the hyperparameter optimisation

- This is designed to show the process of creating and assessing a challenger model without over-engineering the solution

In [ ]:
# run a lightweight Optuna search

study = tune_xgb_model(
    X_train,
    y_train,
    X_test,
    y_test,
    n_trials=3,
)

study.best_value, study.best_params

In [ ]:
# train the tuned model using the best parameters

xgb_tuned = fit_xgb_model(
    X_train,
    y_train,
    X_test,
    y_test,
    params=xgb_params_from_study(study),
    verbose=50,
)

In [ ]:
# score train, test and OOT samples with the tuned model

splits = score_splits(splits, X, xgb_tuned, "pbad_tuned")
train, test, oot = splits["train"], splits["test"], splits["oot"]

### 2.5. Save deployment artefacts

To make the next deployment notebook reusable, I save the fitted model together with the preprocessing metadata required at inference time.

This includes the model object, candidate variables, categorical columns, sparse adverse variables, segment columns, dropped source fields and final feature list.

In [ ]:
# save fitted model and preprocessing metadata for deployment

model_path = save_model_artifacts(
    model=xgb_tuned,
    feature_cols=feature_cols,
    categorical_cols=categorical_cols,
    model_path="models/credit_risk_model.joblib",
)

model_path

### 2.6. Create reusable inference helper

The helper below is written to `src/silvercat/inference.py` so it can be imported from the deployment notebook. It follows a simple AWS-style structure: load artefacts, preprocess input data and return predictions.

In [ ]:
# write local inference helper for the deployment notebook

inference_path = write_inference_helper("src/silvercat/inference.py")

inference_path

#### The workflow in section 2.6 allows us to clone the repo and read in the artefacts for inference directly, which avoids the need to recreate the preprocessing / modelling flow in notebook 3_deployment.

#### In a production environment, this would be handled through persistent storage and a managed deployment process rather than notebook-generated artefacts.

### 2.7. Validate model discrimination performance on OOT

#### power - overall

In [ ]:
# summarise AUC / Gini performance

performance = performance_summary(
    y,
    splits,
    score_cols={
        "baseline": "pbad_baseline",
        "tuned": "pbad_tuned",
    },
)

performance

#### power - by segment

In [ ]:
# create validation bands on OOT

oot_validation = add_validation_bands(oot)

In [ ]:
# gini by term

gini_by_segment(
    oot_validation,
    segment_col="term",
    score_col="pbad_tuned",
)

In [ ]:
# gini by amount

gini_by_segment(
    oot_validation,
    segment_col="loan_amnt_band",
    score_col="pbad_tuned",
)

In [ ]:
# gini by interest rate

gini_by_segment(
    oot_validation,
    segment_col="int_rate_band",
    score_col="pbad_tuned",
)

### Model performance summary

* Overall performance is stable across the development samples, with the baseline model having a Gini of around **0.32 on train**, **0.34 on test** and **0.33 on OOT**, while the tuned model has a Gini of around **0.33 on train**, **0.34 on test** and **0.34 on OOT**

* There is no clear evidence of overfitting. OOT performance is in line with, and slightly above, train performance. This suggests the model is generalising reasonably well to later originations

* The tuned model only marginally improves performance versus the baseline. This suggests that the simple XGBoost specification was already close to optimal for this feature set, and that further gains may require better features rather than heavier tuning

* Segment-level performance is broadly stable by loan amount, with stronger Gini in the larger-loan bands. This suggests the model ranks risk particularly well for larger loans

* Performance by term is acceptable in both groups. The 60-month segment has a much higher bad rate, so it remains an important policy and monitoring segment

* Bad rates increase sharply across interest-rate bands, confirming that interest rate is strongly aligned with underlying credit risk. However, Gini within each band is lower because borrowers inside each pricing band are already more risk-homogeneous

### 2.8. Analyse variable importance

I use SHAP to assess the relative performance contribution of the individual features.

In [ ]:
# take a sample for SHAP

shap_sample_size = min(20_000, len(X_oot))

X_shap = X_oot.sample(
    n=shap_sample_size,
    random_state=42,
)

In [ ]:
# create SHAP explainer for the fitted XGBoost model

explainer = shap.TreeExplainer(xgb_tuned)
shap_values = explainer(X_shap)

In [ ]:
# beeswarm plot

shap.plots.beeswarm(
    shap_values,
    max_display=20,
)

In [ ]:
# features with non-zero contribution

shap_importance_non_zero = shap_importance_table(
    shap_values,
    feature_names=X_shap.columns,
)

shap_importance_non_zero

### Interpretation of SHAP feature importance

The SHAP values show that the key model drivers are core credit-risk characteristics: FICO, recent credit activity, utilisation, available credit and account history. This is consistent with the univariate analysis and suggests that the main risk signals are stable and persistent.

* `fico_avg` is the most important feature. Higher FICO values generally reduce predicted bad risk, while lower FICO values increase it. This is the strongest credit quality signal in the model

* `acc_open_past_24mths` is the second most important feature. Higher values tend to increase predicted bad risk, suggesting that borrowers with more recently opened accounts are riskier

* `purpose_credit_card` has a clear directional effect. The plot suggests that credit-card refinancing loans tend to reduce predicted bad risk relative to other purposes, after controlling for the other model variables

* `percent_bc_gt_75`, `revol_util` and `bc_util` show that higher utilisation is associated with higher predicted bad risk. This is consistent with utilisation acting as a credit pressure or affordability signal

* `mths_since_recent_inq` has an inverse risk relationship. Lower values, meaning more recent credit inquiries, tend to increase predicted bad risk, while a longer time since the last inquiry reduces risk

* `avg_cur_bal`, `bc_open_to_buy` and `tot_hi_cred_lim` generally reduce predicted bad risk when values are high. These variables capture available credit capacity and broader financial depth

* `num_rev_tl_bal_gt_0` and `num_tl_op_past_12m` add further signal around active revolving credit usage and recent account opening

* Home ownership also contributes, with `home_ownership_MORTGAGE` generally associated with lower predicted risk and `home_ownership_RENT` with higher predicted risk

Overall, the model results are consistent with economic theory and are highly explainable, which is important from a model governance perspective.

### 2.9. Model validation scope and next steps

I assess the model's performance using Gini as a proxy for the model risk splitting power on the OOT sample.

Further validation would be required before using the model in a production credit decisioning process. In particular, the next steps would include:

* **Calibration analysis** to compare predicted bad rates with observed bad rates across score bands
* **Probability calibration** if the model is miscalibrated, for example using beta calibration, Platt scaling or isotonic regression
* **Conversion of predicted bad probability into a credit score**, using an agreed scorecard scaling approach
* **Population Stability Index (PSI)** analysis overall, by score band and by key input variable
* **Ongoing monitoring framework**, including monthly bad-rate tracking, score distribution monitoring, segment-level performance and override / policy outcome monitoring

These steps are not implemented here because the purpose of this exercise is to demonstrate model development, performance validation and model explainability rather than a full production deployment framework.

# 3. Discuss the model’s performance and whether it has uncovered additional trends

### Model performance and additional insights

* The model confirms the main findings from the univariate analysis: credit quality, recent credit activity, utilisation and available credit are the strongest risk dimensions

* The main additional insight from the multivariate model is the relative ranking of these signals. Recent credit expansion, especially `acc_open_past_24mths`, is almost as important as FICO once all variables are considered together

* The model also shows that some variables with useful univariate separation have limited incremental value after correlated features are included. This applies to variables such as `account_age_mths`, `total_rev_hi_lim`, `total_bc_limit` and some delinquency-history measures

* Purpose and home ownership add incremental signal after controlling for bureau characteristics, but they should be interpreted as supporting segmentation variables rather than primary policy drivers

* Overall, the model adds value by combining the strongest individual risk signals into a clearer ranking of incremental predictive importance. This supports the use of the model as a risk-ranking tool, with policy overlays for affordability, adverse credit history and risk appetite

# 4. Apply the model to rejected applications (if possible) and discuss how they would score

In [ ]:
# inspect the structure of the rejects dataset

rejects_head = pd.read_csv(
    paths.rejects,
    compression="gzip",
    nrows=10,
    low_memory=False,
)

rejects_head

In [ ]:
rejects_head.columns.tolist()

### Observations from the rejects dataset

* The rejected applications dataset has a much smaller set of variables than the accepted applications dataset

* The rejected dataset includes fields such as `Amount Requested`, `Application Date`, `Loan Title`, `Risk_Score`, `Debt-To-Income Ratio`, `Zip Code`, `State`, `Employment Length` and `Policy Code`

* Most of the variables used by the trained XGBoost model are not available in the rejected dataset. In particular, the rejected data does not contain the detailed bureau variables used by the model, such as utilisation, account history, delinquencies, credit limits, recent inquiries and account mix

* Because of this, the fitted model cannot be applied directly to rejected applications without substantial feature mismatch

* The rejected dataset does include `Risk_Score`, which appears to be a credit score-style variable. However, this was not used in the accepted-applications model, where FICO was represented through `fico_range_low`, `fico_range_high` and `fico_avg`

* It would therefore not be appropriate to substitute `Risk_Score` directly for `fico_avg` without validating that it is defined on the same basis and has the same meaning

* A limited rejected-applicant analysis could still be performed outside the fitted model. For example, rejected applications could be profiled by requested amount, DTI, Risk Score, state and application date, and compared directionally with similar accepted-applicant fields where available

* However, this would be a separate descriptive analysis, not a true application of the trained model

* Overall, the rejected applications dataset is not suitable for direct scoring with the fitted XGBoost model because it lacks the majority of required model inputs. The most defensible conclusion is that rejected applications can be profiled descriptively, but cannot be reliably scored using this model without rebuilding a simpler model based only on variables common to both accepted and rejected applications

# 5. Suggest ways the credit risk policy could be refined based on the model’s outcomes

### Policy refinement opportunities based on the model outcomes

The model can support credit risk policy refinement in two main ways:

* **Decision cut-offs:** using predicted bad probability (`pbad`) to rank applications and assess the trade-off between acceptance rate and observed bad rate
* **Policy rules and overlays:** using the strongest model drivers to define additional referral, decline or affordability review rules

The cut-off should not be based on model performance alone. It should reflect risk appetite, expected profitability, operational capacity, affordability considerations and customer strategy.

In this section, I first assess whether the model ranks risk better than random selection, then simulate simple approval cut-offs on the out-of-time sample. I then discuss how the strongest model variables could inform policy rules.

### Model ranking and decision cut-off simulation

To show how the model could support credit policy refinement, I use the out-of-time sample to test two related questions:

* **Model ranking impact versus random selection:** applications are ordered from highest to lowest predicted bad risk. The cumulative bad capture curve shows what share of observed bad outcomes is captured within the highest-risk share of applications. The random selection line provides a benchmark: if the model curve sits above this line, the model is concentrating bad risk better than random selection

* **Decision cut-off simulation:** applications are ranked from lowest to highest predicted bad risk and grouped into deciles. This shows the trade-off between acceptance rate and observed bad rate as progressively higher-risk bands are accepted

This is not a final production cut-off recommendation. A production decision would also need to consider pricing, expected loss, affordability, operational capacity and commercial return.

In [ ]:
# cumulative bad capture curve on OOT

gains_df = cumulative_bad_capture(oot)

plot_bad_capture_curve(gains_df)

The cumulative bad capture curve shows whether the model concentrates bad outcomes in the highest-risk part of the population. A model curve above the random selection line indicates that the model is ranking applications better than random selection.

In [ ]:
# capture summary at selected high-risk cut-offs

capture_summary(gains_df)

### Decision cut-off simulation

In [ ]:
# summarise approval-rate and bad-rate trade-off

cutoff = cutoff_summary(oot)

cutoff

In [ ]:
# visualise acceptance-rate and bad-rate trade-off

plot_cutoff_tradeoff(cutoff)

The cut-off simulation translates the model ranking into a practical credit policy view. Lower-risk bands can be accepted first, while higher-risk bands could be declined or referred for further review.

The final cut-off would depend on the lender’s risk appetite. For example, a more conservative policy would select a lower acceptance rate with a lower cumulative bad rate, while a higher-growth strategy would accept more applications but tolerate a higher expected bad rate.

### Policy rule implications

The model should be used as the primary ranking tool, while policy rules can act as overlays for risk appetite, affordability and adverse credit history.

The strongest candidates for policy refinement are:

* **FICO / credit quality:** `fico_avg` is the strongest model driver, so policy could include minimum score thresholds or differentiated cut-offs by FICO band

* **Recent credit expansion:** variables such as `acc_open_past_24mths` and `num_tl_op_past_12m` suggest that applicants with high recent account opening may need tighter cut-offs or referral

* **Utilisation and credit pressure:** `percent_bc_gt_75`, `revol_util` and `bc_util` support additional affordability checks or stricter treatment where revolving credit utilisation is high

* **Recent credit-seeking:** `mths_since_recent_inq` and `inq_last_6mths` indicate that recent search activity should be considered in decisioning

* **Credit depth and capacity:** variables such as `bc_open_to_buy`, `avg_cur_bal`, `tot_hi_cred_lim` and `mort_acc` could support more favourable treatment for otherwise borderline applications

* **Adverse-event flags:** sparse markers such as bankruptcies, public records, serious delinquencies or tax liens may be better used as hard policy rules or referral triggers, even if they have low model importance due to low frequency

Overall, the model provides the risk ranking, while policy overlays define how that ranking is translated into accept, decline and refer decisions.